# ShadowLab - Private AI Laboratory
Run this notebook sequentially to deploy the dual-model AI inference pipeline with Cloudflare Tunnel and Google Drive caching.

In [ ]:
# Phase 0: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Phase 1: Installing System Dependencies
!echo "Installing dependencies..."
!apt-get update -qq && apt-get install -y -qq build-essential cmake git wget curl jq libcurl4-openssl-dev 2>/dev/null
!nvcc --version
!nvidia-smi

In [ ]:
# Phase 2: Smart Caching & Compiling llama.cpp
!bash -c '\
ENGINE_DIR="/content/drive/MyDrive/Shadow Lab/engine" ;\
ENGINE_PATH="$ENGINE_DIR/llama-server" ;\
if [ -f "$ENGINE_PATH" ]; then \
  echo "Cached engine found in Google Drive." ;\
  mkdir -p /opt/llama.cpp/build/bin/ ;\
  cp "$ENGINE_PATH" /opt/llama.cpp/build/bin/llama-server ;\
  chmod +x /opt/llama.cpp/build/bin/llama-server ;\
else \
  echo "Engine not found in cache. Compiling from scratch..." ;\
  rm -rf /opt/llama.cpp ;\
  git clone --depth 1 https://github.com/ggerganov/llama.cpp /opt/llama.cpp ;\
  cd /opt/llama.cpp ;\
  cmake -B build -DGGML_CUDA=ON -DLLAMA_CURL=ON -DCMAKE_CUDA_ARCHITECTURES="75" -DCMAKE_BUILD_TYPE=Release ;\
  cmake --build build --config Release -j$(nproc) --target llama-server ;\
  echo "Compilation complete. Caching binary to Google Drive..." ;\
  mkdir -p "$ENGINE_DIR" ;\
  cp build/bin/llama-server "$ENGINE_PATH" ;\
fi ;\
/opt/llama.cpp/build/bin/llama-server --version'

In [ ]:
# Phase 3: Clone Repo & Smart Model Downloader
!echo "Cloning GitHub Repository..."
!rm -rf /opt/Shadow_Lab
!git clone https://github.com/ekanshbfoe/Shadow_Lab.git /opt/Shadow_Lab

!echo "Checking for models in Google Drive cache..."
!mkdir -p "/content/drive/MyDrive/Shadow Lab/models"

!if [ -f "/content/drive/MyDrive/Shadow Lab/models/DeepHat-V1-7B.Q4_K_M.gguf" ]; then echo "DeepHat model found in Drive cache, skipping download."; else echo "Downloading DeepHat..."; wget -q --show-progress -O "/content/drive/MyDrive/Shadow Lab/models/DeepHat-V1-7B.Q4_K_M.gguf" "https://huggingface.co/mradermacher/DeepHat-V1-7B-GGUF/resolve/main/DeepHat-V1-7B.Q4_K_M.gguf"; fi
!if [ -f "/content/drive/MyDrive/Shadow Lab/models/qwen2.5-vl-7b-instruct-q4_k_m.gguf" ]; then echo "Qwen VL model found in Drive cache, skipping download."; else echo "Downloading Qwen VL..."; wget -q --show-progress -O "/content/drive/MyDrive/Shadow Lab/models/qwen2.5-vl-7b-instruct-q4_k_m.gguf" "https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/qwen2.5-vl-7b-instruct-q4_k_m.gguf"; fi
!if [ -f "/content/drive/MyDrive/Shadow Lab/models/mmproj-f16.gguf" ]; then echo "Qwen mmproj found in Drive cache, skipping download."; else echo "Downloading Qwen mmproj..."; wget -q --show-progress -O "/content/drive/MyDrive/Shadow Lab/models/mmproj-f16.gguf" "https://huggingface.co/bartowski/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/mmproj-f16.gguf"; fi
!echo "All models ready in Google Drive cache."

In [ ]:
# Phase 4: Launch Orchestrator
!echo "Launching Orchestrator..."
!pkill -f orchestrator.py || true
!pkill -f llama-server || true
!fuser -k 8000/tcp 2>/dev/null || true
!fuser -k 8081/tcp 2>/dev/null || true
!sleep 1

%cd /opt/Shadow_Lab/colab_engine/orchestrator
!pip install -q -r requirements.txt

!nohup python3 orchestrator.py \
  --listen-host 0.0.0.0 \
  --listen-port 8000 \
  --backend-port 8081 \
  --llama-server /opt/llama.cpp/build/bin/llama-server \
  --model-dir "/content/drive/MyDrive/Shadow Lab/models" \
  > /var/log/orchestrator.log 2>&1 &

!sleep 3
!for i in $(seq 1 20); do curl -sf http://localhost:8000/health && echo '' && break || sleep 1; done
!echo "Orchestrator Ready."

In [ ]:
# Phase 5: Cloudflare Tunnel
import subprocess
import time
import re
import os

os.system('wget -q -O /usr/local/bin/cloudflared "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"')
os.system('chmod +x /usr/local/bin/cloudflared')
os.system('pkill -f cloudflared || true')

# The magic fix: start_new_session=True completely detaches the process from the cell
with open('/var/log/cloudflared.log', 'w') as log_file:
    subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

print("Extracting tunnel URL...")
tunnel_url = None
for _ in range(30):
    time.sleep(2)
    try:
        with open("/var/log/cloudflared.log", "r") as f:
            log_content = f.read()
            match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_content)
            if match:
                tunnel_url = match.group(0)
                break
    except FileNotFoundError:
        pass

if tunnel_url:
    print("\n============================================================")
    print(f"  PUBLIC ENDPOINT: {tunnel_url}")
    print("============================================================\n")
    with open("/content/tunnel_url.txt", "w") as f:
        f.write(tunnel_url)
    # Also save to Google Drive for Windows sync
    drive_sync = "/content/drive/MyDrive/Shadow Lab/tunnel_url.txt"
    os.makedirs(os.path.dirname(drive_sync), exist_ok=True)
    with open(drive_sync, "w") as f:
        f.write(tunnel_url)
    print(f"  Saved to Google Drive for Windows sync.")
else:
    print("FATAL: Could not extract tunnel URL")
    

In [ ]:
# Phase 6: Keepalive and Monitoring
import time, requests, subprocess
from IPython.display import display, Javascript

display(Javascript('''
  setInterval(() => { google.colab.kernel.invokeFunction("keepalive", [], {}); }, 60000);
'''))

HEALTH_URL = "http://localhost:8000/health"
CHECK_INTERVAL = 120

print("🟢 Monitoring started. Keep this cell running.")
while True:
    try:
        r = requests.get(HEALTH_URL, timeout=10)
        gpu = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total,temperature.gpu", "--format=csv,noheader,nounits"],
            text=True
        ).strip()
        mem_used, mem_total, temp = gpu.split(", ")
        status = "🟢" if r.status_code == 200 else "🟡"
        print(f"{status} [{time.strftime('%H:%M:%S')}] API: {r.status_code} | GPU: {mem_used}/{mem_total} MB | Temp: {temp}°C")
    except Exception as e:
        print(f"🔴 [{time.strftime('%H:%M:%S')}] Health check failed: {e}")
    time.sleep(CHECK_INTERVAL)
